In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', '00-setup'))
from spark_session import get_spark, output_path, stop_and_wait
from seed_data import load_all, register_views
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = get_spark("11-date-time")
dfs = register_views(spark)
emp  = dfs["employees"]
ee   = dfs["emp_events"]
lr   = dfs["leave_requests"]
sal  = dfs["salary_history"]

# ── Section 1: current_date, current_timestamp, now ──────────────────────────
# current_date()      → DATE (no time component)
# current_timestamp() → TIMESTAMP (driver-local wall clock)
# now()               → SQL alias for current_timestamp()
from pyspark.sql.functions import current_date, current_timestamp

print("=== Section 1: current_date / current_timestamp / now ===")
spark.sql("SELECT current_date(), current_timestamp(), now()").show()

# ── Section 2: datediff, date_add, date_sub ───────────────────────────────────
# datediff(end, start) → integer days between two dates.
# date_add(date, n)    → date + n days.
# date_sub(date, n)    → date - n days.
print("=== Section 2: datediff / date_add – tenure and next review ===")
emp.withColumn("tenure_days",  F.datediff(F.current_date(), F.col("hire_date"))) \
   .withColumn("tenure_years", F.round(F.col("tenure_days") / 365.25, 1)) \
   .select("emp_id", "first_name", "hire_date", "tenure_days", "tenure_years") \
   .orderBy("hire_date").show(5)

emp.withColumn("next_review", F.date_add(F.col("hire_date"), 365)) \
   .select("emp_id", "hire_date", "next_review").show(3)

# ── Section 3: months_between and add_months ─────────────────────────────────
# months_between(end, start) → fractional months (double).
# add_months(date, n)        → date shifted n calendar months.
print("=== Section 3: months_between – fractional tenure in months ===")
emp.withColumn("months_tenured", F.round(F.months_between(F.current_date(), F.col("hire_date")), 1)) \
   .select("emp_id", "first_name", "hire_date", "months_tenured").show(5)

# ── Section 4: date_format and date_trunc ─────────────────────────────────────
# date_format: format a date/timestamp as a string using Java SimpleDateFormat patterns.
# date_trunc:  truncate to the start of the given unit (month, week, year, day, hour…).
print("=== Section 4: date_format – hire_month and hire_year strings ===")
emp.withColumn("hire_month", F.date_format(F.col("hire_date"), "yyyy-MM")) \
   .withColumn("hire_year",  F.date_format(F.col("hire_date"), "yyyy")) \
   .select("emp_id", "hire_date", "hire_month", "hire_year").show(5)

print("=== Section 4: date_trunc – month_start and week_start ===")
emp.withColumn("month_start", F.date_trunc("month", F.col("hire_date"))) \
   .select("emp_id", "hire_date", "month_start").show(3)
emp.withColumn("week_start",  F.date_trunc("week",  F.col("hire_date"))) \
   .select("emp_id", "hire_date", "week_start").show(3)

# ── Section 5: year, month, dayofmonth, dayofweek, quarter, weekofyear ────────
# Convenience extractors — equivalent to date_format but return integers.
# dayofweek: 1=Sunday … 7=Saturday (Spark default, not ISO).
print("=== Section 5: date part extractors ===")
emp.withColumn("yr",  F.year("hire_date")) \
   .withColumn("mo",  F.month("hire_date")) \
   .withColumn("dom", F.dayofmonth("hire_date")) \
   .withColumn("dow", F.dayofweek("hire_date")) \
   .withColumn("qtr", F.quarter("hire_date")) \
   .select("emp_id", "hire_date", "yr", "mo", "dom", "dow", "qtr").show(5)

# ── Section 6: to_date and to_timestamp parsing ───────────────────────────────
# to_date / to_timestamp: parse string columns using explicit format masks.
# NULL is returned silently if the string doesn't match the format.
print("=== Section 6: to_date – parse strings to dates ===")
spark.createDataFrame([("2024-01-15",), ("15/01/2024",)], ["date_str"]) \
     .withColumn("parsed_iso", F.to_date("date_str", "yyyy-MM-dd")) \
     .withColumn("parsed_dmy", F.to_date("date_str", "dd/MM/yyyy")) \
     .show()

print("=== Section 6: date_format on TIMESTAMP column ===")
ee.withColumn("ts_formatted", F.date_format(F.col("event_ts"), "yyyy-MM-dd HH:mm:ss")) \
  .select("event_id", "event_ts", "ts_formatted").show(3)

# ── Section 7: last_day, next_day ─────────────────────────────────────────────
# last_day(date)              → last calendar day of that month.
# next_day(date, "Monday")    → first occurrence of that weekday AFTER the date.
print("=== Section 7: last_day / next_day ===")
emp.withColumn("end_of_hire_month", F.last_day("hire_date")) \
   .withColumn("next_monday",       F.next_day("hire_date", "Monday")) \
   .select("emp_id", "hire_date", "end_of_hire_month", "next_monday").show(5)

# ── Section 8: Timestamp arithmetic (gap between events) ─────────────────────
# No native TIMESTAMPDIFF in PySpark; cast TIMESTAMP to LONG (Unix epoch seconds)
# and subtract to get gap in seconds, then divide to desired unit.
print("=== Section 8: gap_minutes between consecutive employee events ===")
w_emp_time = Window.partitionBy("emp_id").orderBy("event_ts")
ee.withColumn("prev_ts", F.lag("event_ts", 1).over(w_emp_time)) \
  .withColumn("gap_minutes",
      F.round(
          (F.col("event_ts").cast("long") - F.col("prev_ts").cast("long")) / 60, 1
      )
  ).select("event_id", "emp_id", "event_type", "event_ts", "gap_minutes").show(10)
# prev_ts is NULL for the first event per employee → gap_minutes is also NULL (expected)

# ── Section 9: Cohort month from hire_date ────────────────────────────────────
# Group employees by the month they were hired (cohort analysis).
# date_format("yyyy-MM") produces a sortable string key.
print("=== Section 9: hire cohort sizes by month ===")
emp.withColumn("cohort", F.date_format("hire_date", "yyyy-MM")) \
   .groupBy("cohort").count() \
   .orderBy("cohort").show()

# ── Section 10: Detect future dates (data flaw) ───────────────────────────────
# Comparing a date column to current_date() flags records with hire_date > today.
# Data flaw: emp 41 has hire_date = 2025-08-01 (future date in the seed data).
print("=== Section 10: employees with future hire_date (data flaw) ===")
emp.filter(F.col("hire_date") > F.current_date()) \
   .select("emp_id", "first_name", "hire_date").show()
# Expected: emp 41 (hire_date = 2025-08-01)

# ── Section 11: Leave duration in days ───────────────────────────────────────
# datediff(end, start) + 1 gives inclusive day count (both endpoints counted).
print("=== Section 11: leave request duration (inclusive days) ===")
lr.withColumn("duration_days", F.datediff(F.col("end_date"), F.col("start_date")) + 1) \
  .select("request_id", "emp_id", "leave_type", "start_date", "end_date", "duration_days") \
  .show()

# ── Section 12: Cross-year leave (start year != end year) ────────────────────
# A Boolean expression comparing year() of start vs. end date identifies requests
# that span a calendar year boundary — relevant for accrual calculations.
print("=== Section 12: leave requests spanning a calendar year boundary ===")
lr.withColumn("cross_year", F.year("start_date") != F.year("end_date")) \
  .filter(F.col("cross_year")) \
  .select("request_id", "emp_id", "start_date", "end_date").show()
# Expected: request_id 9 and 10 for emp 16 (Dec 2023 → Jan 2024)

stop_and_wait(spark)